# 🧠 EX57: Bounding Box Parsing

```
results[0].boxes
 ├── .xyxy   [N,4]  pixel coords [x1,y1,x2,y2]
 ├── .xywh   [N,4]  pixel [cx,cy,w,h]
 ├── .xyxyn  [N,4]  normalized [0,1]
 ├── .cls    [N,1]  class int
 └── .conf   [N,1]  confidence float
```

**GPU→CPU rule:** Always `.cpu()` before `.numpy()`:
```python
boxes.xyxy.cpu().numpy()  # ✅
boxes.xyxy.numpy()        # ❌ RuntimeError on CUDA tensors
```

## 🔗 Links
- [[EX56_Prediction_Sources_TH]] | [[EX58_Model_Export_TH]]


In [ ]:
# Back up or checkpoint this section of code before starting to modify the large file.
import gc, os
import cv2, numpy as np, torch
import matplotlib.pyplot as plt
from ultralytics import YOLO
from solution import parse_yolo_results
%matplotlib inline

device = "0" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Device: {device}")

img = np.zeros((480, 640, 3), dtype=np.uint8)
cv2.rectangle(img, (50, 80), (200, 300), (200, 200, 0), -1)
cv2.rectangle(img, (350, 100), (580, 380), (0, 200, 200), -1)
cv2.imwrite("bb_test.jpg", img)

print("\n--- AUDIT & INSPECTION START ---")
model = YOLO("yolo11n.pt")
results = model.predict(source="bb_test.jpg", verbose=False)
boxes = results[0].boxes

print(f"  Detections: {len(boxes)}")
print(f"  Device:     {boxes.xyxy.device}")
print(f"  .xyxy shape:{boxes.xyxy.shape}  (N×4)")
print(f"  .cls  shape:{boxes.cls.shape}")
print(f"  .conf shape:{boxes.conf.shape}")

if len(boxes):
    cls_arr  = boxes.cls.cpu().numpy().astype(int).flatten()
    conf_arr = boxes.conf.cpu().numpy().flatten()
    xyxy_arr = boxes.xyxy.cpu().numpy()
    norm_arr = boxes.xyxyn.cpu().numpy()
    print("\n  Detections:")
    for i,(cls,conf,xyxy,norm) in enumerate(zip(cls_arr,conf_arr,xyxy_arr,norm_arr)):
        print(f"    #{i+1} {model.names[cls]:<18} conf={conf:.4f}")
        print(f"         xyxy ={[f'{v:.1f}' for v in xyxy]}")
        print(f"         norm ={[f'{v:.3f}' for v in norm]}")
else:
    print("  No detections (expected for blank synthetic image).")

parsed = parse_yolo_results("yolo11n.pt", "bb_test.jpg")
print(f"\n  solution.parse_yolo_results() → {len(parsed)} detections")
print("--- AUDIT & INSPECTION END ---")

plt.figure(figsize=(8,5))
plt.imshow(cv2.cvtColor(results[0].plot(), cv2.COLOR_BGR2RGB))
plt.title("Bounding Box Parsing — Annotated"); plt.axis("off"); plt.show()

del model
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
if os.path.exists("bb_test.jpg"): os.remove("bb_test.jpg")
